# Building a Basic Chatbot with LangGraph

In this notebook, we will build a basic chatbot (local LLM calls only) using `LangGraph`, so you can get familiar with the APIs and its structure.

## Recommended Hardware

This notebook can run on the following hardware or remote resources.

✅ AMD Instinct™ Accelerators  
✅ AMD Radeon™ RX/PRO Graphics Cards  
✅ AMD EPYC™ Processors  
✅ AMD Ryzen™ (AI) Processors  

[![Open in AMD Developer Cloud](https://img.shields.io/badge/Open_in_AMD_Developer_Cloud-000000?logo=amd&logoSize=auto)](https://amd-ai-academy.com/github/AMDResearch/aup-ai-tutorials/blob/main/ai-agents/01-chatbot.ipynb)  

## Software Environment

Install ROCm on your system.

| Linux | Windows |
|-------|---------|
| [Install PyTorch](https://rocm.docs.amd.com/projects/install-on-linux/en/latest/install/quick-start.html) | [PyTorch on Windows](https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html)|
| [Install Docker container](https://amdresearch.github.io/aup-ai-tutorials//env/env-gpu.html) | |

## Goals

- Build a basic chatbot using LangGraph and a local LLM.
- Define a `StateGraph` with nodes and edges.
- Stream and display chatbot responses.

### Install Dependencies

Install the package dependencies needed for this notebook.

First, get the `aup_config.py` script locally with the package dependencies.

In [ ]:
!wget https://raw.githubusercontent.com/AMDResearch/aup-ai-tutorials/refs/heads/main/ai-agents/aup_config.py

Install the dependencies. This step may take a few minutes and only needs to be done once.

In [ ]:
from aup_config import aup_setup
aup_setup()

## Building a Chatbot with LangGraph

### Import Libraries

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from IPython.display import display

from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

### Define `StateGraph`

We start by creating our `State` class that is used to build the `StateGraph` object. The `StateGraph` defines the structure of our chatbot as a set of nodes and edges. We add `nodes` to represent the LLM and functions/tools our chatbot can use, and `edges` to specify the transitions between nodes.

In [ ]:
class State(TypedDict):
    """
    Messages have the type 'list'.
    The `add_messages` function defines how this state key should be updated
    (in this case, it appends messages to the list, rather than overwriting them)
    """
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

The graph handles two important tasks
- Each node will receive the current `State` as input and update the state as output.
- Updates to the `messages` attribute will be appended to the existing list. This is called [reducer functions](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers)

### Define LLM Model

We use `model_provider='openai'` so you can use a different LLM serving framework if needed.

In [ ]:
llm = init_chat_model(
    model='qwen3:8b',
    model_provider='openai',
    base_url='http://localhost:11434/v1/',
    api_key='abc-123'
)

### Add a Node

Now, we can add a node to the graph. `Nodes` represent units of work.

Let us add the LLM call into a simple node

Note how the `chatbot` function takes the `State` as input and returns a dictionary containing an updated `messages` list.

The `.add_node` takes at least two arguments
1. Unique node name
2. Object that will be called when the node is used.

In [ ]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}


graph_builder.add_node("chatbot", chatbot)

### Add an `entry` point

LangGraph works with directed graphs, so we need to specify the starting point for our graph each time it is run. Note that we use the keyword `START` and the node name `chatbot`.

In [ ]:
graph_builder.add_edge(START, "chatbot")

### Add an `exit` point

Similarly, we need to indicate the node where the graph finishes execution. Note that we use the keyword `END` and the node name `chatbot`.

In [ ]:
graph_builder.add_edge("chatbot", END)

### Compile the Graph

Once all the nodes and edges are added, we need to compile our graph.

In [ ]:
graph = graph_builder.compile()

### Visualize the graph

We can visualize the graph to make sure the nodes and edges are correct. 

In [ ]:
display({"text/vnd.mermaid": graph.get_graph().draw_mermaid()}, raw=True)

### Run the Chatbot

With the graph compiled, we can now run it. Let us create a function that takes a user string and returns the response from the model.

In [ ]:
def stream_graph_updates(user_input: str):
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

In [ ]:
stream_graph_updates('What are Large Language Models?')

## Exercises for the Reader

- Try a different model from the Ollama library and compare the responses.
- Which answers seem more accurate? Which are more detailed?

## Conclusions

In this notebook, you built a basic chatbot using a local LLM. You also observed that without retrieval, the model may hallucinate on domain-specific questions. This motivates adding RAG in the next steps.

## References/Resources

<div class="alert alert-block alert-info">
<ul>
  <li><a href="https://www.langchain.com/langgraph">LangGraph</a></li>
</ul>
</div>

---

[AMD University Program](https://www.amd.com/aup).

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.

SPDX-License-Identifier: MIT